## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

## TODAY:

- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors

### PART A: Divide our documents into chunks

In [ ]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from App.config import azure_endpoint, api_version, headers

In [ ]:
# price is a factor for our test company, so we're going to use a low cost model

db_name = 'vector_db'
load_dotenv(override=True)
ollama_api_key = os.getenv('OLLAMA_API_KEY')
gemini_api_key = os.getenv('GOOGLE_API_KEY')

if ollama_api_key:
    print(f'Ollama API key found')
else:
    print('Ollama API key not set')

if gemini_api_key:
    print(f'Gemini API key found')
else:
    print('Gemini API key not set')

gpt_model = 'gpt-oss:120b'
gpt_oss_tokenizer = "o200k_harmony"
gemini_model = 'gemini-3.1-flash-lite'

In [ ]:
# How many characters in all the documents ?

knowledge_base_path = "week5/knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f'Found {len(files)} files in the knowledge base')

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as fs:
        entire_knowledge_base += fs.read()
        entire_knowledge_base += "\n\n"

print(f'Total characters in knowledge base: {len(entire_knowledge_base):,}')


In [ ]:
# How many tokens in all the documents?

# Getting encoding for gpt-oss model

gpt_encoding = tiktoken.get_encoding(gpt_oss_tokenizer)
gpt_tokens = gpt_encoding.encode(entire_knowledge_base)
token_count = len(gpt_tokens)
print(f'Total tokens for model {gpt_model}: {token_count:,}')


In [ ]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob('week5/knowledge-base/*')

documents = []

for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob='**/*.md', loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata['doc_type'] = doc_type
        documents.append(doc)

print(f'Loaded {len(documents)} documents')

In [ ]:
documents[0:5]

In [ ]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

In [ ]:
chunks[100]

### PART B: Make vectors and store in Chroma

In Week 3, you set up a Hugging Face account and got an HF_TOKEN

At this point, you might want to add it to your `.env` file and run `load_dotenv(override=True)`

(This actually shouldn't be required).